# Batch Normalisation from Scratch: Why It Works and When It Doesn't

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/batch_normalisation.ipynb)

Implement BatchNorm, LayerNorm, InstanceNorm, and GroupNorm from scratch in NumPy. Visualise how each normalisation changes activation distributions.

**Blog post:** [sesen.ai/blog/batch-normalisation-from-scratch](https://sesen.ai/blog/batch-normalisation-from-scratch)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

## The Dramatic Effect of BatchNorm on Training

A 5-layer CNN trained with and without BatchNorm. Without BN, lr=0.1 diverges. With BN, lr=0.5 (50x higher) converges to 98.6%.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = datasets.MNIST('data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('data', train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000)

def make_cnn(use_bn=False):
    """Build a 5-layer CNN, optionally with BatchNorm after each conv."""
    layers = []
    channels = [(1, 8, 5), (8, 16, 3), (16, 32, 3), (32, 64, 3), (64, 64, 3)]
    for ni, nf, ks in channels:
        layers.append(nn.Conv2d(ni, nf, ks, padding=ks//2, stride=2, bias=not use_bn))
        if use_bn:
            layers.append(nn.BatchNorm2d(nf))
        layers.append(nn.ReLU())
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)]
    return nn.Sequential(*layers)

def train_and_evaluate(model, lr, epochs=5):
    optimiser = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    results = []
    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            loss = F.cross_entropy(model(images), labels)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_loader:
                correct += (model(images).argmax(1) == labels).sum().item()
        results.append(correct / len(test_data) * 100)
        print(f"  Epoch {epoch+1}: {results[-1]:.1f}%")
    return results

In [ ]:
# Without BN: lr=0.01 works, lr=0.1 DIVERGES
print("Without BatchNorm (lr=0.01):")
torch.manual_seed(42)
acc_safe = train_and_evaluate(make_cnn(use_bn=False), lr=0.01)

print("\nWithout BatchNorm (lr=0.1) — diverges:")
torch.manual_seed(42)
acc_diverge = train_and_evaluate(make_cnn(use_bn=False), lr=0.1)

print("\nWith BatchNorm (lr=0.5) — stable and fast:")
torch.manual_seed(42)
acc_bn = train_and_evaluate(make_cnn(use_bn=True), lr=0.5)

In [ ]:
# Plot training comparison
fig, ax = plt.subplots(figsize=(10, 6))
epochs = range(1, 6)

ax.plot(epochs, acc_safe, 'o-', color='#3b82f6', linewidth=2, markersize=8,
        label='No BN (lr=0.01)')
ax.plot(epochs, acc_diverge, 'o-', color='#ef4444', linewidth=2, markersize=8,
        label='No BN (lr=0.1) — diverges')
ax.plot(epochs, acc_bn, 'o-', color='#22c55e', linewidth=2, markersize=8,
        label='With BN (lr=0.5)')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('BatchNorm Enables 50x Higher Learning Rate', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## Building BatchNorm from Scratch

In [ ]:
class BatchNorm2d_NumPy:
    """Batch normalisation for 2D inputs (B, C, H, W)."""

    def __init__(self, num_features, momentum=0.1, eps=1e-5):
        self.eps = eps
        self.momentum = momentum
        # Learnable parameters
        self.gamma = np.ones((1, num_features, 1, 1))
        self.beta = np.zeros((1, num_features, 1, 1))
        # Running statistics (for inference)
        self.running_mean = np.zeros((1, num_features, 1, 1))
        self.running_var = np.ones((1, num_features, 1, 1))

    def forward(self, x, training=True):
        if training:
            # Compute mean and variance over batch and spatial dims
            mean = x.mean(axis=(0, 2, 3), keepdims=True)
            var = x.var(axis=(0, 2, 3), keepdims=True)
            # Update running statistics
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            mean = self.running_mean
            var = self.running_var

        # Normalise, then scale and shift
        x_hat = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

## Verifying Against PyTorch

In [ ]:
np.random.seed(42)
x_np = np.random.randn(4, 3, 4, 4).astype(np.float32)

# NumPy version
bn_np = BatchNorm2d_NumPy(3)
out_np = bn_np.forward(x_np, training=True)

# PyTorch version
bn_pt = nn.BatchNorm2d(3, momentum=0.1, eps=1e-5)
with torch.no_grad():
    bn_pt.weight.fill_(1.0)
    bn_pt.bias.fill_(0.0)
bn_pt.train()
out_pt = bn_pt(torch.tensor(x_np)).detach().numpy()

print(f"Max difference: {np.max(np.abs(out_np - out_pt)):.2e}")
print(f"Match: {np.allclose(out_np, out_pt, atol=1e-5)}")

## Visualising Activation Distributions

Watch what happens to activation distributions across layers with and without BatchNorm.

In [ ]:
def get_activation_stats(model, data_loader, n_batches=10):
    """Collect activation means and stds from each conv/bn layer."""
    stats = {}
    hooks = []

    def hook_fn(name):
        def fn(module, input, output):
            if name not in stats:
                stats[name] = {'means': [], 'stds': []}
            stats[name]['means'].append(output.detach().mean().item())
            stats[name]['stds'].append(output.detach().std().item())
        return fn

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.BatchNorm2d, nn.ReLU)):
            hooks.append(module.register_forward_hook(hook_fn(name)))

    model.eval()
    with torch.no_grad():
        for i, (images, _) in enumerate(data_loader):
            if i >= n_batches: break
            model(images)

    for h in hooks: h.remove()
    return stats

In [ ]:
# Collect activations from trained models
torch.manual_seed(42)
model_no_bn = make_cnn(use_bn=False)
# Quick train for activation stats
opt = torch.optim.SGD(model_no_bn.parameters(), lr=0.01, momentum=0.9)
model_no_bn.train()
for i, (images, labels) in enumerate(train_loader):
    if i >= 100: break
    loss = F.cross_entropy(model_no_bn(images), labels)
    opt.zero_grad(); loss.backward(); opt.step()

torch.manual_seed(42)
model_with_bn = make_cnn(use_bn=True)
opt = torch.optim.SGD(model_with_bn.parameters(), lr=0.5, momentum=0.9)
model_with_bn.train()
for i, (images, labels) in enumerate(train_loader):
    if i >= 100: break
    loss = F.cross_entropy(model_with_bn(images), labels)
    opt.zero_grad(); loss.backward(); opt.step()

stats_no_bn = get_activation_stats(model_no_bn, test_loader)
stats_bn = get_activation_stats(model_with_bn, test_loader)

# Plot activation histograms
# Get ReLU layer names
relu_names_no_bn = [n for n in stats_no_bn if 'relu' in n.lower() or isinstance(
    dict(model_no_bn.named_modules()).get(n), nn.ReLU)]
relu_names_bn = [n for n in stats_bn if 'relu' in n.lower() or isinstance(
    dict(model_with_bn.named_modules()).get(n), nn.ReLU)]

# Collect actual activations for histograms
def get_activations(model, data_loader):
    activations = []
    hooks = []
    def hook_fn(module, input, output):
        activations.append(output.detach().cpu().numpy().flatten()[:5000])
    for name, module in model.named_modules():
        if isinstance(module, nn.ReLU):
            hooks.append(module.register_forward_hook(hook_fn))
    model.eval()
    with torch.no_grad():
        images, _ = next(iter(data_loader))
        model(images)
    for h in hooks: h.remove()
    return activations

acts_no_bn = get_activations(model_no_bn, test_loader)
acts_bn = get_activations(model_with_bn, test_loader)

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i in range(5):
    if i < len(acts_no_bn):
        axes[0, i].hist(acts_no_bn[i], bins=50, color='#ef4444', alpha=0.7, density=True)
        axes[0, i].set_title(f'Layer {i+1} (no BN)', fontsize=10)
        axes[0, i].set_xlim(-1, 5)
    if i < len(acts_bn):
        axes[1, i].hist(acts_bn[i], bins=50, color='#22c55e', alpha=0.7, density=True)
        axes[1, i].set_title(f'Layer {i+1} (with BN)', fontsize=10)
        axes[1, i].set_xlim(-1, 5)

axes[0, 0].set_ylabel('Without BN', fontsize=12)
axes[1, 0].set_ylabel('With BN', fontsize=12)
plt.suptitle('Activation Distributions Across Layers', fontsize=14)
plt.tight_layout()
plt.show()

## The Normalisation Family

The difference between norms is *which dimensions they normalise over*.

### LayerNorm: Normalise Over Features

In [ ]:
class LayerNorm_NumPy:
    """Layer normalisation: normalise over (C, H, W) for each sample."""

    def __init__(self, eps=1e-5):
        self.eps = eps
        self.gamma = 1.0
        self.beta = 0.0

    def forward(self, x):
        # Normalise over channels and spatial dims (axes 1, 2, 3)
        mean = x.mean(axis=(1, 2, 3), keepdims=True)
        var = x.var(axis=(1, 2, 3), keepdims=True)
        x_hat = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

### InstanceNorm: Normalise Each Channel Independently

In [ ]:
class InstanceNorm_NumPy:
    """Instance normalisation: normalise over (H, W) per sample per channel."""

    def __init__(self, num_features, eps=1e-5):
        self.eps = eps
        self.gamma = np.ones((1, num_features, 1, 1))
        self.beta = np.zeros((1, num_features, 1, 1))

    def forward(self, x):
        # Normalise over spatial dims only (axes 2, 3)
        mean = x.mean(axis=(2, 3), keepdims=True)
        var = x.var(axis=(2, 3), keepdims=True)
        x_hat = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

### GroupNorm: A Practical Compromise

In [ ]:
class GroupNorm_NumPy:
    """Group normalisation: normalise within channel groups."""

    def __init__(self, num_groups, num_channels, eps=1e-5):
        self.num_groups = num_groups
        self.eps = eps
        self.gamma = np.ones((1, num_channels, 1, 1))
        self.beta = np.zeros((1, num_channels, 1, 1))

    def forward(self, x):
        B, C, H, W = x.shape
        G = self.num_groups
        # Reshape to (B, G, C//G, H, W)
        x = x.reshape(B, G, C // G, H, W)
        mean = x.mean(axis=(2, 3, 4), keepdims=True)
        var = x.var(axis=(2, 3, 4), keepdims=True)
        x_hat = (x - mean) / np.sqrt(var + self.eps)
        x_hat = x_hat.reshape(B, C, H, W)
        return self.gamma * x_hat + self.beta

### Comparing All Four Norms

In [ ]:
# Compare outputs on the same input
np.random.seed(42)
x_test = np.random.randn(4, 8, 4, 4).astype(np.float32)

bn = BatchNorm2d_NumPy(8)
ln = LayerNorm_NumPy()
inn = InstanceNorm_NumPy(8)
gn = GroupNorm_NumPy(num_groups=4, num_channels=8)

out_bn = bn.forward(x_test)
out_ln = ln.forward(x_test)
out_in = inn.forward(x_test)
out_gn = gn.forward(x_test)

print("Output statistics (mean, std) for sample 0, channel 0:")
for name, out in [("BatchNorm", out_bn), ("LayerNorm", out_ln),
                   ("InstanceNorm", out_in), ("GroupNorm", out_gn)]:
    print(f"  {name:13s}: mean={out[0, 0].mean():.4f}, std={out[0, 0].std():.4f}")

## RunningBatchNorm for Small Batches

Standard BatchNorm fails with batch sizes of 1-4. This variant uses smoothed running statistics even during training.

In [ ]:
class RunningBatchNorm_NumPy:
    """Running BN: uses exponential moving averages during training too.

    Solves the small-batch problem by never relying on batch statistics alone.
    """

    def __init__(self, num_features, momentum=0.1, eps=1e-5):
        self.eps = eps
        self.momentum = momentum
        self.gamma = np.ones((1, num_features, 1, 1))
        self.beta = np.zeros((1, num_features, 1, 1))
        self.running_mean = np.zeros((1, num_features, 1, 1))
        self.running_var = np.ones((1, num_features, 1, 1))
        self.step = 0

    def forward(self, x, training=True):
        if training:
            batch_mean = x.mean(axis=(0, 2, 3), keepdims=True)
            batch_var = x.var(axis=(0, 2, 3), keepdims=True)
            self.step += 1
            self.running_mean = ((1 - self.momentum) * self.running_mean
                                  + self.momentum * batch_mean)
            self.running_var = ((1 - self.momentum) * self.running_var
                                 + self.momentum * batch_var)
            # Use debiased running stats instead of raw batch stats
            debias = 1 - (1 - self.momentum) ** self.step
            mean = self.running_mean / debias
            var = self.running_var / debias
        else:
            mean = self.running_mean
            var = self.running_var

        x_hat = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

# Demo: RunningBatchNorm handles small batches
rbn = RunningBatchNorm_NumPy(3)
# Simulate small batch (size 2)
x_small = np.random.randn(2, 3, 4, 4).astype(np.float32)
out_rbn = rbn.forward(x_small, training=True)
print(f"RunningBatchNorm output shape: {out_rbn.shape}")
print(f"Output mean: {out_rbn.mean():.4f}, std: {out_rbn.std():.4f}")

## Exercises

1. **Small batch experiment** — Train `make_cnn(use_bn=True)` with batch_size=2 for 3 epochs. Then replace `nn.BatchNorm2d` with `nn.GroupNorm(num_groups=4, ...)`. Compare accuracy.

2. **Placement order** — Train one model with Conv → BN → ReLU and another with Conv → ReLU → BN. Compare accuracy after 5 epochs.

3. **Verify other norms against PyTorch** — Use `torch.nn.LayerNorm`, `torch.nn.InstanceNorm2d`, and `torch.nn.GroupNorm` to verify each NumPy implementation matches.

4. **Eval mode bug** — Train a model with BatchNorm, then evaluate *without* calling `model.eval()`. Compare accuracy to the correct eval. How big is the gap?

5. **Weight decay on BN params** — Train with `weight_decay=0.01` applied to all parameters, then train again with weight decay only on conv weights (not BN gamma/beta). Does it matter?

## References

- Ioffe, S. & Szegedy, C. (2015). [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift.](https://arxiv.org/abs/1502.03167)
- Santurkar, S. et al. (2018). [How Does Batch Normalization Actually Help?](https://arxiv.org/abs/1805.11604)
- Ba, J.L., Kiros, J.R. & Hinton, G.E. (2016). [Layer Normalization.](https://arxiv.org/abs/1607.06450)
- Wu, Y. & He, K. (2018). [Group Normalization.](https://arxiv.org/abs/1803.08494)
- fast.ai course: [Practical Deep Learning for Coders, Lesson 10](https://course.fast.ai/).